# Supplementary Material — 1. Generate Probabilities

Companion notebook to *Polynomial Multiplication for Post-Match Soccer Outcome Probabilities*.

For each of the five 2015/16 European top leagues (Premier League, Bundesliga,
La Liga, Serie A, Ligue 1) this notebook generates outcome and exact-score
probability distributions under five approaches and saves them for evaluation
in `02_evaluation.ipynb`. Only the 1822 matches with complete records (both
StatsBomb xG data and Bet365 odds) are included; this excludes 3 Ligue 1
matches missing from StatsBomb and 1 Ligue 1 match without a betting-odds entry,
leaving 376 complete Ligue 1 records.

| Approach | Source |
|---|---|
| `poisson_binomial`  | Polynomial multiplication with per-shot xG (Ruiz et al., 2015) |
| `double_poisson`    | Two independent Poissons with team-xG sums as rates |
| `bivariate_poisson` | Karlis & Ntzoufras (2003); shared component λ₃ fitted by MLE |
| `dixon_coles`       | Dixon & Coles (1997) low-score adjustment; ρ fitted by MLE |
| `betting_odds`      | Pre-match Bet365 implied probabilities (football-data.co.uk) |

In [33]:
import sys
from pathlib import Path

# Re-use the analysis modules shipped with the main project
sys.path.insert(0, str(Path('..').resolve() / 'modules'))

import pandas as pd
from tqdm.notebook import tqdm

from data_loading import (
    load_competition_data,
    load_football_data_odds,
    merge_odds_with_matches,
)
from probability_generation import (
    fit_bivariate_poisson_lambda3,
    fit_dixon_coles_rho,
    generate_poisson_binomial_probs,
    generate_double_poisson_probs,
    generate_bivariate_poisson_probs,
    generate_dixon_coles_probs,
    generate_betting_odds_probs,
    generate_poisson_binomial_score_dists,
    generate_double_poisson_score_dists,
    generate_bivariate_poisson_score_dists,
    generate_dixon_coles_score_dists,
    save_probabilities,
)

INPUT_DIR  = Path('..') / '..' / 'data' / 'input_data'
ODDS_DIR   = Path('..') / '..' / 'data' / 'odds'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_GOALS = 10  # per-team goals cap

## 1. League configuration

The five-leagues × football-data.co.uk file mapping.

In [34]:
LEAGUES = [
    {'slug': 'epl_1516',        'odds_code': 'E0',  'comp_name': 'England - Premier League'},
    {'slug': 'bundesliga_1516', 'odds_code': 'D1',  'comp_name': 'Germany - 1. Bundesliga'},
    {'slug': 'laliga_1516',     'odds_code': 'SP1', 'comp_name': 'Spain - La Liga'},
    {'slug': 'seriea_1516',     'odds_code': 'I1',  'comp_name': 'Italy - Serie A'},
    {'slug': 'ligue1_1516',     'odds_code': 'F1',  'comp_name': 'France - Ligue 1'},
]

pd.DataFrame(LEAGUES)

,slug,odds_code,comp_name
0,epl_1516,E0,England - Premier League
1,bundesliga_1516,D1,Germany - 1. Bundesliga
2,laliga_1516,SP1,Spain - La Liga
3,seriea_1516,I1,Italy - Serie A
4,ligue1_1516,F1,France - Ligue 1


## 2. Generate predictions per league

For each league we (i) load the StatsBomb-derived match summary, (ii) merge
Bet365 implied probabilities by `(date, home, away)`, (iii) fit the
Bivariate-Poisson λ₃ and Dixon–Coles ρ globally on observed scores, and
(iv) generate outcome and score parquet files for all five approaches.

In [35]:
fitted_params = []

for cfg in tqdm(LEAGUES, total=len(LEAGUES), desc='Leagues'):
    slug = cfg['slug']
    print(f'\n=== {slug} ===')

    # 1) Load and merge betting odds; retain only complete records
    data = load_competition_data(INPUT_DIR / f'{slug}.parquet')
    odds = load_football_data_odds(ODDS_DIR / f"{cfg['odds_code']}.csv",
                                   competition_name=cfg['comp_name'],
                                   date_format='%Y-%m-%d')
    data = merge_odds_with_matches(data, odds)
    data = data[data['betting_p_home'].notna()].reset_index(drop=True)
    print(f'  Using {len(data)} complete records.')

    # 2) Fit dependence parameters once per league (MLE on observed scores)
    lambda3 = fit_bivariate_poisson_lambda3(data, max_goals=MAX_GOALS)
    rho     = fit_dixon_coles_rho(data, max_goals=MAX_GOALS)
    print(f'  Bivariate-Poisson λ₃ = {lambda3:.4f}   |   Dixon–Coles ρ = {rho:+.4f}')
    fitted_params.append({'league': slug, 'lambda3_BP': lambda3, 'rho_DC': rho})

    # 3) Outcome probabilities
    save_probabilities(generate_poisson_binomial_probs(data, max_goals=MAX_GOALS),
                       OUTPUT_DIR, slug, 'poisson_binomial', 'outcomes')
    save_probabilities(generate_double_poisson_probs(data, max_goals=MAX_GOALS),
                       OUTPUT_DIR, slug, 'double_poisson', 'outcomes')
    save_probabilities(generate_bivariate_poisson_probs(data, max_goals=MAX_GOALS, lambda3=lambda3),
                       OUTPUT_DIR, slug, 'bivariate_poisson', 'outcomes')
    save_probabilities(generate_dixon_coles_probs(data, max_goals=MAX_GOALS, rho=rho),
                       OUTPUT_DIR, slug, 'dixon_coles', 'outcomes')
    save_probabilities(generate_betting_odds_probs(data),
                       OUTPUT_DIR, slug, 'betting_odds', 'outcomes')

    # 4) Score distributions (no betting-odds equivalent — odds give outcome only)
    save_probabilities(generate_poisson_binomial_score_dists(data, max_goals=MAX_GOALS),
                       OUTPUT_DIR, slug, 'poisson_binomial', 'scores')
    save_probabilities(generate_double_poisson_score_dists(data, max_goals=MAX_GOALS),
                       OUTPUT_DIR, slug, 'double_poisson', 'scores')
    save_probabilities(generate_bivariate_poisson_score_dists(data, max_goals=MAX_GOALS, lambda3=lambda3),
                       OUTPUT_DIR, slug, 'bivariate_poisson', 'scores')
    save_probabilities(generate_dixon_coles_score_dists(data, max_goals=MAX_GOALS, rho=rho),
                       OUTPUT_DIR, slug, 'dixon_coles', 'scores')

print('\nFitted dependence parameters:')
print(pd.DataFrame(fitted_params).to_string(index=False))

Leagues:   0%|          | 0/5 [00:00<?, ?it/s]


=== epl_1516 ===
Betting odds matched for 380/380 matches.
  Using 380 complete records.
  Bivariate-Poisson λ₃ = 0.0144   |   Dixon–Coles ρ = -0.0775
Saved 380 rows → output/epl_1516_poisson_binomial_outcomes.parquet
Saved 380 rows → output/epl_1516_double_poisson_outcomes.parquet
Saved 380 rows → output/epl_1516_bivariate_poisson_outcomes.parquet
Saved 380 rows → output/epl_1516_dixon_coles_outcomes.parquet
Saved 380 rows → output/epl_1516_betting_odds_outcomes.parquet
Saved 380 rows → output/epl_1516_poisson_binomial_scores.parquet
Saved 380 rows → output/epl_1516_double_poisson_scores.parquet
Saved 380 rows → output/epl_1516_bivariate_poisson_scores.parquet
Saved 380 rows → output/epl_1516_dixon_coles_scores.parquet

=== bundesliga_1516 ===
Betting odds matched for 306/306 matches.
  Using 306 complete records.
  Bivariate-Poisson λ₃ = 0.0002   |   Dixon–Coles ρ = -0.0647
Saved 306 rows → output/bundesliga_1516_poisson_binomial_outcomes.parquet
Saved 306 rows → output/bundesliga_1

## 3. Saved files

In [4]:
rows = []
for f in sorted(OUTPUT_DIR.glob('*.parquet')):
    rows.append({'file': f.name, 'rows': len(pd.read_parquet(f)),
                 'size_kb': round(f.stat().st_size / 1024, 1)})
pd.DataFrame(rows)

,file,rows,size_kb
0,bundesliga_1516_betting_odds_outcomes.parquet,306,15.9
1,bundesliga_1516_bivariate_poisson_outcomes.par...,306,17.5
2,bundesliga_1516_bivariate_poisson_scores.parquet,306,752.4
3,bundesliga_1516_dixon_coles_outcomes.parquet,306,17.4
4,bundesliga_1516_dixon_coles_scores.parquet,306,753.2
5,bundesliga_1516_double_poisson_outcomes.parquet,306,17.5
6,bundesliga_1516_double_poisson_scores.parquet,306,753.2
7,bundesliga_1516_poisson_binomial_outcomes.parquet,306,17.5
8,bundesliga_1516_poisson_binomial_scores.parquet,306,656.0
9,epl_1516_betting_odds_outcomes.parquet,380,19.2
